# Scoring a clustering without reading its IDs

A lab clustered 400 cells whose types it later learned from staining, and wants to know how
well each clustering recovered the types. Cluster IDs are arbitrary, some methods mark
noise points with $-1$, and the types are strings, so `pred == y` means nothing here. You
will build the table of types against clusters, the adjusted Rand index, and the matched
accuracy, all from that table.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np
from itertools import permutations

rng = np.random.default_rng(5)
y = np.array(["T"] * 200 + ["B"] * 100 + ["NK"] * 60 + ["mono"] * 40)

# Clustering 1: a density method. Each type mostly lands in one cluster, about 8% of the
# cells land somewhere else, and -1 marks cells it called noise.
cluster_of = {"T": 7, "B": 2, "NK": 4, "mono": 9}
density = np.array([cluster_of[t] for t in y])
moved = rng.random(len(y)) < 0.08
density[moved] = rng.choice([7, 2, 4, 9, -1], size=moved.sum())

# Clustering 2: k-means with k = 2, which split the T cells in half and put everything else
# with the second half.
kmeans2 = np.where(np.arange(len(y)) < 100, 0, 1)

# Clustering 3: every cell in one cluster.
lump = np.zeros(len(y), dtype=int)
print({str(t): int((y == t).sum()) for t in np.unique(y)})

## Part 1 (coding task): The table of types against clusters

Write `contingency(y, pred)`. It returns `(table, classes, clusters)`, where `classes` is the
sorted array of distinct values in `y`, `clusters` the sorted array of distinct values in
`pred`, and `table[i, j]` the number of items whose type is `classes[i]` and whose cluster
is `clusters[j]`. The values can be any labels: strings, negative numbers, or IDs with gaps.

In [ ]:
def contingency(y, pred):
    """Counts of each type (rows) in each cluster (columns)."""
    # TODO
    raise NotImplementedError

In [ ]:
t, cl, cu = contingency(np.array(["b", "a", "a", "c", "b", "a"]), np.array([5, -1, 5, 5, 2, -1]))
assert list(cl) == ["a", "b", "c"] and list(cu) == [-1, 2, 5], f"classes {list(cl)} and clusters {list(cu)} should be the sorted distinct labels"
assert np.array_equal(t, [[2, 0, 1], [0, 1, 1], [0, 0, 1]]), f"wrong table:\n{t}\nrows are types, columns are clusters, both in sorted order"
t, cl, cu = contingency(y, density)
assert t.shape == (4, 5) and t.sum() == len(y), f"the cells' table should be 4 by 5 and hold all {len(y)} cells"
assert np.array_equal(t.sum(axis=1), [100, 60, 200, 40]), "row sums should be the type counts B, NK, T, mono"
print(t)
print("Part 1 passed")

## Part 2 (coding task): The adjusted Rand index

Write `ari(y, pred)` from the table. With $n_{ij}$ the table entries, $a_i$ the row sums,
$b_j$ the column sums and $n$ the number of items, count pairs of items:

- $S = \sum_{i,j} \binom{n_{ij}}{2}$, the pairs that both labelings put together,
- $A = \sum_i \binom{a_i}{2}$ and $B = \sum_j \binom{b_j}{2}$,
- $E = A B / \binom{n}{2}$, the value $S$ would take on average if the clusters were
  shuffled at random with their sizes kept.

Then $\text{ARI} = (S - E) / \big(\tfrac12 (A + B) - E\big)$. You may assume the denominator is
not zero.

In [ ]:
def ari(y, pred):
    """ARI of pred against the true types y."""
    # TODO
    raise NotImplementedError

In [ ]:
ya = np.array([0, 0, 0, 1, 1, 1, 1, 2, 2])
pa = np.array([0, 0, 1, 1, 1, 2, 2, 2, 2])
assert np.isclose(ari(ya, pa), 11 / 65), f"expected 11/65 = 0.1692 on the nine-item example, got {ari(ya, pa):.4f}"
assert np.isclose(ari(ya, 10 - 3 * ya), 1.0), "renaming the clusters should leave the ARI at 1"
assert np.isclose(ari(pa, ya), ari(ya, pa)), "the ARI should not depend on which labeling comes first"
assert np.isclose(ari(y, lump), 0.0), "putting every cell in one cluster should score 0"
print(f"ARI: density method {ari(y, density):.3f}, k-means with k = 2 {ari(y, kmeans2):.3f}, one cluster {ari(y, lump):.3f}")
print("Part 2 passed")

## Part 3 (coding task): Matched accuracy

Write `matched_accuracy(y, pred)`. Pair the types with the clusters one to one so that as
many items as possible have a cluster paired with their type, and return that number of
items divided by the number of items. When there are more clusters than types, some
clusters are left unpaired, and their items count as wrong. The same goes for types when
there are fewer clusters.

The tables here are small, so trying every one-to-one pairing is fine:
`permutations(range(c), r)` lists every way to give the $r$ rows distinct columns out of
$c \ge r$. If there are more types than clusters, transpose the table first, so its
rows are always the shorter side and `r` is never larger than `c`.

In [ ]:
def matched_accuracy(y, pred):
    """Share of items matched under the best one-to-one pairing of types and clusters."""
    # TODO
    raise NotImplementedError

In [ ]:
def from_table(T):
    # Labels for items laid out as a table: row i is the type, column j the cluster.
    T = np.asarray(T)
    rows, cols = np.nonzero(T)
    counts = T[rows, cols]
    return np.repeat(rows, counts), np.repeat(cols, counts)

T3 = [[10, 9, 0], [8, 0, 1], [0, 1, 7]]
got = matched_accuracy(*from_table(T3))
assert np.isclose(got, 24 / 36), f"expected 24/36 = 0.667, got {got:.3f}: each cluster pairs with one type only, and the pairing that starts from the largest count is not the best"
got = matched_accuracy(*from_table([[6, 5, 0], [0, 5, 4]]))
assert np.isclose(got, 11 / 20), f"two types and three clusters: expected 11/20, got {got:.3f}"
got = matched_accuracy(*from_table([[6, 0], [5, 5], [0, 4]]))
assert np.isclose(got, 11 / 20), f"three types and two clusters: expected 11/20, got {got:.3f}: each cluster can be paired with only one type"
print(f"{'clustering':<22}{'matched acc':>12}{'ARI':>8}")
for name, pred in [("density method", density), ("k-means, k = 2", kmeans2), ("one cluster", lump)]:
    print(f"{name:<22}{matched_accuracy(y, pred):>12.3f}{ari(y, pred):>8.3f}")
print("Part 3 passed")

## Solution

### Part 1: The table of types against clusters

In [ ]:
def contingency(y, pred):
    """Counts of each type (rows) in each cluster (columns)."""
    classes, row = np.unique(np.asarray(y), return_inverse=True)
    clusters, col = np.unique(np.asarray(pred), return_inverse=True)
    table = np.zeros((len(classes), len(clusters)), dtype=int)
    np.add.at(table, (row, col), 1)
    return table, classes, clusters

**Why it works.** `np.unique(..., return_inverse=True)` maps any labels, strings and $-1$ included, to the
positions 0 to $m - 1$ of the sorted distinct values, so the labels never serve as indices
themselves. `np.add.at` then adds 1 at each (type, cluster) position, counting repeated
positions correctly, which plain fancy-index assignment would not.

### Part 2: The adjusted Rand index

In [ ]:
def ari(y, pred):
    """ARI of pred against the true types y."""
    table, _, _ = contingency(y, pred)
    pairs = lambda m: m * (m - 1) / 2
    S = pairs(table).sum()
    A = pairs(table.sum(axis=1)).sum()
    B = pairs(table.sum(axis=0)).sum()
    E = A * B / pairs(table.sum())
    return float((S - E) / ((A + B) / 2 - E))

**Why it works.** A pair of items is either together or apart in each labeling. $S$ counts the pairs that are
together in both, and $A$ and $B$ count the pairs together in each labeling. If one labeling
were a random relabeling with the same cluster sizes, a pair together in $y$ would be together
in `pred` with probability $B / \binom{n}{2}$, so $S$ would average $E$. Subtracting $E$ and
dividing by the largest value $S - E$ can take, reached when the two labelings agree, gives 1
for identical partitions and about 0 for unrelated ones. The plain Rand index skips the
subtraction and gives 0.34 on the one-cluster labeling here, which says nothing about how
useless it is.

### Part 3: Matched accuracy

In [ ]:
def matched_accuracy(y, pred):
    """Share of items matched under the best one-to-one pairing of types and clusters."""
    table, _, _ = contingency(y, pred)
    if table.shape[0] > table.shape[1]:
        table = table.T
    r, c = table.shape
    best = max(table[np.arange(r), list(cols)].sum() for cols in permutations(range(c), r))
    return float(best / table.sum())

**Why it works.** A one-to-one pairing picks at most one cell from each row and each column of the table, and
the matched items are the sum of the picked cells. Putting the shorter side on the rows and
trying every injective choice of columns covers every pairing, including the ones that leave
clusters or types unpaired. Greedy pairing fails on the first check table: giving the 10 to
the first type takes column 0, which holds 8 of the second type's 9 cells.

The last table shows why both scores are worth having. The one-cluster labeling gets a matched
accuracy of 0.5, the share of the largest type, for finding nothing, and its ARI of 0 says
so. The k-means split scores 0.5 as well and a small ARI, since it found only that some T
cells differ from the rest. The density method scores high on both.